# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/subikshasrig/FlyrankMLInternship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [17]:
from huggingface_hub import login
from google.colab import userdata
from datasets import load_dataset
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

HF_TOKEN = userdata.get("HF_TOKEN")

login(token=HF_TOKEN)

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    streaming=True,
    split="train"
)

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

table_path = f"{rel}/fact_content_daily_performance/**/*.parquet"

print("Setup complete.")

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Setup complete.


## 1. Question

*The research question and the decision it supports.*

In [ ]:
overview = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date,
    COUNT(DISTINCT client_hash_id) AS client_count,
    COUNT(DISTINCT content_hash_id) AS content_count,
    COUNT(DISTINCT report_date) AS reporting_days
FROM read_parquet('{table_path}')
""").df()

display(overview)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [ ]:
availability = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN gsc_data_available THEN 1 ELSE 0 END) AS rows_with_gsc,
    SUM(CASE WHEN ga4_data_available THEN 1 ELSE 0 END) AS rows_with_ga4,
    SUM(CASE WHEN gsc_impressions > 0 THEN 1 ELSE 0 END) AS rows_with_impressions,
    SUM(CASE WHEN gsc_clicks > 0 THEN 1 ELSE 0 END) AS rows_with_clicks,
    SUM(CASE WHEN ga4_sessions > 0 THEN 1 ELSE 0 END) AS rows_with_sessions,
    SUM(CASE WHEN sessions_organic > 0 THEN 1 ELSE 0 END) AS rows_with_organic_sessions,
    SUM(CASE WHEN sessions_ai > 0 THEN 1 ELSE 0 END) AS rows_with_ai_sessions
FROM read_parquet('{table_path}')
""").df()

display(availability)

observation_history = con.sql(f"""
SELECT
    content_hash_id,
    COUNT(*) AS days_observed
FROM read_parquet('{table_path}')
GROUP BY content_hash_id
""").df()

display(observation_history["days_observed"].describe())

for days in [30, 60, 90, 180, 365]:
    print(
        f">= {days} days: "
        f"{(observation_history['days_observed'] >= days).sum():,}"
    )

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

feature_start = pd.Timestamp("2026-05-06")
feature_end = pd.Timestamp("2026-06-02")

future_start = pd.Timestamp("2026-06-03")
future_end = pd.Timestamp("2026-06-30")

feature_data = con.sql(f"""
SELECT
    content_hash_id,
    SUM(gsc_impressions) AS impressions_28d,
    SUM(gsc_clicks) AS clicks_28d,
    CASE
        WHEN SUM(gsc_impressions) > 0
        THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
        ELSE 0
    END AS ctr_28d,
    AVG(gsc_avg_position) AS avg_position_28d,
    SUM(ga4_sessions) AS sessions_28d,
    SUM(ga4_engaged_sessions) AS engaged_sessions_28d,
    SUM(ga4_total_engagement_sec) AS engagement_seconds_28d,
    SUM(sessions_organic) AS organic_sessions_28d,
    SUM(sessions_direct) AS direct_sessions_28d,
    SUM(sessions_referral) AS referral_sessions_28d,
    SUM(sessions_social) AS social_sessions_28d,
    SUM(sessions_paid) AS paid_sessions_28d,
    SUM(sessions_ai) AS ai_sessions_28d,
    SUM(scroll_events) AS scroll_events_28d,
    COUNT(*) AS observed_days_28d
FROM read_parquet('{table_path}')
WHERE report_date BETWEEN '{feature_start.date()}' AND '{feature_end.date()}'
GROUP BY content_hash_id
""").df()

future_data = con.sql(f"""
SELECT
    content_hash_id,
    SUM(gsc_impressions) AS future_impressions_28d,
    SUM(gsc_clicks) AS future_clicks_28d,
    SUM(ga4_sessions) AS future_sessions_28d,
    COUNT(*) AS future_observed_days
FROM read_parquet('{table_path}')
WHERE report_date BETWEEN '{future_start.date()}' AND '{future_end.date()}'
GROUP BY content_hash_id
""").df()

model_data = feature_data.merge(
    future_data,
    on="content_hash_id",
    how="inner"
)

model_data = model_data[
    (model_data["observed_days_28d"] >= 14) &
    (model_data["future_observed_days"] >= 14) &
    (model_data["impressions_28d"] >= 20)
].copy()

model_data["decline_label"] = (
    model_data["future_impressions_28d"]
    <= 0.70 * model_data["impressions_28d"]
).astype(int)

feature_columns = [
    "impressions_28d",
    "clicks_28d",
    "ctr_28d",
    "avg_position_28d",
    "sessions_28d",
    "engaged_sessions_28d",
    "engagement_seconds_28d",
    "organic_sessions_28d",
    "referral_sessions_28d",
    "direct_sessions_28d",
    "paid_sessions_28d",
    "social_sessions_28d",
    "ai_sessions_28d",
    "scroll_events_28d",
    "observed_days_28d"
]

X = model_data[feature_columns]
y = model_data["decline_label"]

future_variables = [
    "future_impressions_28d",
    "future_clicks_28d",
    "future_sessions_28d",
    "future_observed_days",
    "decline_label"
]

leakage_features = [
    col for col in feature_columns
    if col in future_variables
]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model.fit(X_train, y_train)

print("Research Question:")
print(
    "Can recent search visibility and engagement signals identify "
    "content that is likely to experience a meaningful decline in "
    "search performance over the following 28 days?"
)

print("\nFeature window:")
print(f"{feature_start.date()} → {feature_end.date()}")

print("\nFuture prediction window:")
print(f"{future_start.date()} → {future_end.date()}")

print("\nModel dataset:")
print(f"Rows: {len(model_data):,}")
print(f"Columns: {len(feature_columns)}")

print("\nLabel distribution:")
print(model_data["decline_label"].value_counts())

print("\nLabel proportions:")
print(model_data["decline_label"].value_counts(normalize=True))

print("\nLeakage check:")
print(f"Features used: {len(feature_columns)}")
print(f"Future variables excluded: {future_variables}")

if len(leakage_features) == 0:
    print("✓ No future-period variables are included as model features.")
else:
    print("WARNING: Potential leakage detected:", leakage_features)

print("\nValidation split:")
print(f"Training rows: {len(X_train):,}")
print(f"Test rows:     {len(X_test):,}")

print("\nTraining positive rate:")
print(y_train.mean())

print("\nTest positive rate:")
print(y_test.mean())

print("\nBaseline:")
print("Majority class:", y_train.mode()[0])

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.50).astype(int)

majority_class = y_train.mode()[0]
baseline_pred = np.full(len(y_test), majority_class)

results = pd.DataFrame({
    "Model": [
        "Majority-class baseline",
        "Logistic Regression"
    ],
    "Accuracy": [
        accuracy_score(y_test, baseline_pred),
        accuracy_score(y_test, y_pred)
    ],
    "Precision": [
        precision_score(y_test, baseline_pred, zero_division=0),
        precision_score(y_test, y_pred, zero_division=0)
    ],
    "Recall": [
        recall_score(y_test, baseline_pred, zero_division=0),
        recall_score(y_test, y_pred, zero_division=0)
    ],
    "F1": [
        f1_score(y_test, baseline_pred, zero_division=0),
        f1_score(y_test, y_pred, zero_division=0)
    ],
    "ROC-AUC": [
        np.nan,
        roc_auc_score(y_test, y_prob)
    ],
    "PR-AUC": [
        np.nan,
        average_precision_score(y_test, y_prob)
    ]
})

display(results)

conf_matrix = confusion_matrix(y_test, y_pred)

print("\nConfusion Matrix:")
print(conf_matrix)

threshold_rows = []

for threshold in np.arange(0.10, 0.95, 0.05):
    threshold_pred = (y_prob >= threshold).astype(int)

    precision = precision_score(
        y_test,
        threshold_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        threshold_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        threshold_pred,
        zero_division=0
    )

    flagged = threshold_pred.mean() * 100

    threshold_rows.append({
        "Threshold": round(threshold, 2),
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Flagged content (%)": flagged
    })

threshold_results = pd.DataFrame(threshold_rows)

best_threshold_row = threshold_results.loc[
    threshold_results["F1"].idxmax()
]

best_threshold = float(best_threshold_row["Threshold"])

print("\nThreshold Analysis:")
display(threshold_results)

print(f"Best F1 threshold: {best_threshold:.2f}")
print(f"Precision: {best_threshold_row['Precision']:.4f}")
print(f"Recall: {best_threshold_row['Recall']:.4f}")
print(f"F1: {best_threshold_row['F1']:.4f}")
print(
    f"Content flagged: "
    f"{best_threshold_row['Flagged content (%)']:.2f}%"
)

print("\nDiscrimination Performance:")
print(
    f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}"
)
print(
    f"PR-AUC: "
    f"{average_precision_score(y_test, y_prob):.4f}"
)

coefficients = pd.DataFrame({
    "Feature": feature_columns,
    "Coefficient": model.named_steps["classifier"].coef_[0]
}).sort_values(
    "Coefficient",
    ascending=False
)

print("\nModel Feature Coefficients:")
display(coefficients)

results.to_csv(
    "model_results.csv",
    index=False
)

threshold_results.to_csv(
    "threshold_results.csv",
    index=False
)

coefficients.to_csv(
    "model_coefficients.csv",
    index=False
)

plt.figure(figsize=(8, 5))
plt.plot(
    threshold_results["Threshold"],
    threshold_results["F1"],
    marker="o"
)
plt.axvline(
    best_threshold,
    linestyle="--"
)
plt.xlabel("Decision Threshold")
plt.ylabel("F1 Score")
plt.title("F1 Score Across Decision Thresholds")
plt.tight_layout()
plt.show()

## 5. Limitations

*What this work cannot claim.*

In [ ]:
limitations = pd.DataFrame({
    "Limitation": [
        "Observational data",
        "No causal inference",
        "Analytical decline definition",
        "Fixed 28-day windows",
        "Incomplete measurement availability",
        "Sparse AI-referral data",
        "Feature multicollinearity",
        "Simple model form",
        "Validation design",
        "Public-safe aggregation",
        "Recommendations are decision-support"
    ],
    "What it means": [
        "The analysis observes relationships between measured signals and later search-performance outcomes.",
        "The model cannot establish that any measured signal caused a subsequent change in search performance.",
        "Decline is defined analytically as future impressions at or below 70% of feature-window impressions.",
        "The 28-day feature and prediction windows are modeling choices.",
        "GSC and GA4 data are not available for every daily record.",
        "AI-referral sessions are relatively sparse and are not the primary prediction target.",
        "Traffic variables are related to one another, so individual coefficients should be interpreted directionally.",
        "Logistic Regression captures relatively simple relationships.",
        "The experiment uses a stratified 80/20 train-test split rather than repeated rolling-origin validation.",
        "Only anonymized identifiers and aggregate performance signals are used.",
        "Predictions prioritize human review and do not guarantee that an intervention will improve performance."
    ]
})

display(limitations)

print("\nClaims this analysis can support:")

supported_claims = [
    "The model identified patterns associated with subsequent observed impression declines.",
    f"The model achieved an ROC-AUC of {roc_auc_score(y_test, y_prob):.3f}.",
    f"The model achieved a PR-AUC of {average_precision_score(y_test, y_prob):.3f}.",
    f"The model achieved an F1 score of {f1_score(y_test, y_pred):.3f} at the 0.50 threshold.",
    f"The selected {best_threshold:.2f} threshold provides a recall of {best_threshold_row['Recall']:.3f}.",
    "The output can be used as a directional content-review prioritization tool."
]

for claim in supported_claims:
    print("•", claim)

print("\nClaims this analysis cannot support:")

unsupported_claims = [
    "The model proves how Google's ranking algorithm works.",
    "A feature causes rankings, impressions or clicks to change.",
    "Refreshing flagged content will definitely improve performance.",
    "The model directly predicts Google's ranking position.",
    "The model proves that an algorithm update caused a decline.",
    "The model guarantees future search performance."
]

for claim in unsupported_claims:
    print("•", claim)

limitations.to_csv(
    "limitations_honest_framing.csv",
    index=False
)

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [ ]:
model_data = model_data.copy()

model_data["decline_probability"] = model.predict_proba(
    model_data[feature_columns]
)[:, 1]

impression_q75 = model_data["impressions_28d"].quantile(0.75)
impression_q50 = model_data["impressions_28d"].quantile(0.50)

model_data["visibility_tier"] = np.select(
    [
        model_data["impressions_28d"] >= impression_q75,
        model_data["impressions_28d"] >= impression_q50
    ],
    [
        "High visibility",
        "Moderate visibility"
    ],
    default="Lower visibility"
)

model_data["position_tier"] = np.select(
    [
        model_data["avg_position_28d"] <= 10,
        model_data["avg_position_28d"] <= 20
    ],
    [
        "Top 10 average position",
        "Positions 11–20"
    ],
    default="Position 21+"
)

model_data["risk_tier"] = np.select(
    [
        model_data["decline_probability"] >= 0.70,
        model_data["decline_probability"] >= best_threshold
    ],
    [
        "High risk",
        "Review risk"
    ],
    default="Lower risk"
)

def generate_reason(row):
    if (
        row["decline_probability"] >= 0.70
        and row["impressions_28d"] >= impression_q75
    ):
        return "High decline risk + high search visibility"

    if (
        row["decline_probability"] >= 0.70
        and row["avg_position_28d"] <= 20
    ):
        return "High decline risk + relatively strong search position"

    if (
        row["decline_probability"] >= best_threshold
        and row["impressions_28d"] >= impression_q75
    ):
        return "Review risk + high search visibility"

    if (
        row["decline_probability"] >= best_threshold
        and row["avg_position_28d"] <= 20
    ):
        return "Review risk + meaningful search position"

    if row["decline_probability"] >= best_threshold:
        return "Review risk based on recent performance signals"

    if (
        row["impressions_28d"] >= impression_q75
        and row["avg_position_28d"] <= 10
    ):
        return "Strong visibility — protect and monitor"

    return "Lower predicted decline risk — monitor"

model_data["reason_code"] = model_data.apply(
    generate_reason,
    axis=1
)

def generate_action(row):
    if (
        row["decline_probability"] >= 0.70
        and row["impressions_28d"] >= impression_q75
    ):
        return "Prioritize refresh review"

    if (
        row["decline_probability"] >= 0.70
        and row["avg_position_28d"] <= 20
    ):
        return "Investigate and consider refresh"

    if (
        row["decline_probability"] >= best_threshold
        and row["impressions_28d"] >= impression_q75
    ):
        return "Review content and metadata"

    if row["decline_probability"] >= best_threshold:
        return "Add to content review queue"

    if (
        row["impressions_28d"] >= impression_q75
        and row["avg_position_28d"] <= 10
    ):
        return "Protect current performance and monitor"

    return "Monitor"

model_data["recommended_action"] = model_data.apply(
    generate_action,
    axis=1
)

visibility_score = np.log1p(
    model_data["impressions_28d"]
)

visibility_score = (
    visibility_score / visibility_score.max()
)

model_data["priority_score"] = (
    0.75 * model_data["decline_probability"]
    + 0.25 * visibility_score
)

ranked_recommendations = (
    model_data
    .sort_values(
        [
            "priority_score",
            "decline_probability",
            "impressions_28d"
        ],
        ascending=[False, False, False]
    )
    .reset_index(drop=True)
)

ranked_recommendations["priority_rank"] = (
    ranked_recommendations.index + 1
)

recommendation_columns = [
    "priority_rank",
    "content_hash_id",
    "decline_probability",
    "priority_score",
    "risk_tier",
    "visibility_tier",
    "position_tier",
    "impressions_28d",
    "clicks_28d",
    "ctr_28d",
    "avg_position_28d",
    "reason_code",
    "recommended_action"
]

recommendation_table = ranked_recommendations[
    recommendation_columns
].copy()

display(recommendation_table.head(25))

priority_distribution = (
    ranked_recommendations["risk_tier"]
    .value_counts()
    .rename_axis("Risk tier")
    .reset_index(name="Content count")
)

priority_distribution["Percentage"] = (
    priority_distribution["Content count"]
    / len(ranked_recommendations)
    * 100
)

display(priority_distribution)

action_distribution = (
    ranked_recommendations["recommended_action"]
    .value_counts()
    .rename_axis("Recommended action")
    .reset_index(name="Content count")
)

action_distribution["Percentage"] = (
    action_distribution["Content count"]
    / len(ranked_recommendations)
    * 100
)

display(action_distribution)

review_queue = ranked_recommendations[
    ranked_recommendations["decline_probability"]
    >= best_threshold
].copy()

high_value_queue = ranked_recommendations[
    (
        ranked_recommendations["decline_probability"]
        >= best_threshold
    )
    &
    (
        ranked_recommendations["impressions_28d"]
        >= impression_q75
    )
].copy()

print(
    f"Review queue: {len(review_queue):,} "
    f"({len(review_queue) / len(ranked_recommendations):.2%})"
)

print(
    f"High-value review queue: {len(high_value_queue):,} "
    f"({len(high_value_queue) / len(ranked_recommendations):.2%})"
)

display(
    high_value_queue[
        recommendation_columns
    ].head(25)
)

action_playbook = pd.DataFrame({
    "Priority": [1, 2, 3, 4],
    "Condition": [
        "High decline risk + high visibility",
        "High decline risk + moderate/lower visibility",
        "Review risk",
        "Lower predicted risk"
    ],
    "Recommended action": [
        "Prioritize refresh review",
        "Investigate content and search signals",
        "Review content and metadata",
        "Protect current performance and monitor"
    ]
})

display(action_playbook)

recommendation_table.to_csv(
    "ranked_content_recommendations.csv",
    index=False
)

priority_distribution.to_csv(
    "recommendation_risk_distribution.csv",
    index=False
)

action_distribution.to_csv(
    "recommendation_action_distribution.csv",
    index=False
)

high_value_queue[
    recommendation_columns
].to_csv(
    "high_value_review_queue.csv",
    index=False
)

action_playbook.to_csv(
    "content_action_playbook.csv",
    index=False
)

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    threshold_results["Threshold"],
    threshold_results["F1"],
    marker="o"
)

plt.axvline(
    best_threshold,
    linestyle="--"
)

plt.xlabel("Decision Threshold")
plt.ylabel("F1 Score")
plt.title("F1 Score Across Decision Thresholds")
plt.tight_layout()
plt.savefig(
    "threshold_f1_curve.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()


plt.figure(figsize=(8, 5))

plt.bar(
    results["Model"],
    results["F1"]
)

plt.ylabel("F1 Score")
plt.title("Model vs Majority-Class Baseline")
plt.tight_layout()
plt.savefig(
    "model_vs_baseline.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()


plt.figure(figsize=(8, 5))

plt.bar(
    action_distribution["Recommended action"],
    action_distribution["Percentage"]
)

plt.ylabel("Content (%)")
plt.xlabel("Recommended action")
plt.title("Recommended Action Distribution")
plt.xticks(
    rotation=25,
    ha="right"
)
plt.tight_layout()
plt.savefig(
    "recommended_action_distribution.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()


plt.figure(figsize=(8, 5))

plt.hist(
    ranked_recommendations["decline_probability"],
    bins=20
)

plt.axvline(
    best_threshold,
    linestyle="--"
)

plt.xlabel("Predicted Decline Probability")
plt.ylabel("Content Count")
plt.title("Distribution of Predicted Decline Risk")
plt.tight_layout()
plt.savefig(
    "decline_probability_distribution.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
